In [78]:
import jieba
import random
import pkuseg
import zhconv
import json
import numpy as np


## 读取训练数据

In [39]:
all_data = []
with open("data/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        one_line = json.loads(line)
        all_data.append([one_line['sentence'], one_line['label_desc'][5:]])



In [40]:
all_data[0]

['上课时学生手机响个不停，老师一怒之下把手机摔了，家长拿发票让老师赔，大家怎么看待这种事？', 'edu']

In [41]:
len(all_data)

53360

In [42]:
# 清洗数据
def full2half(string):
    """
    将传入数据进行全角到半角的转换
    :param string: str 传入的字符串
    :return: str 转换完毕的字符串
    """
    rstring = ""
    for char in string:
        inside_code=ord(char)
        if inside_code == 12288:
            #全角空格直接转换
            inside_code = 32
            rstring += chr(inside_code)
        elif inside_code >= 65281 and inside_code <= 65374:
            #全角字符（除空格） 根据关系转化
            inside_code -= 65248
            rstring += chr(inside_code)
        else:
            rstring += chr(inside_code)
    return rstring


In [43]:
test_string = "abc欢迎来到万门大学ｐｙｔｈｏｎ"

In [44]:
full2half(test_string)

'abc欢迎来到万门大学python'

In [45]:
def data_cleaning(words, cleaning_parameters):
    """
    将传入数据进行数据清洗
    :param words: str 传入的字符串
    :param cleaning_parameters: list 传入的控制开关列表
    :return: str 清洗完毕的字符串
    """
    if cleaning_parameters[0]:
        words = zhconv.convert(words, 'zh-cn')
    if cleaning_parameters[1]:
        words = words.lower()
    if cleaning_parameters[2]:
        words = "".join(words.split())
    if cleaning_parameters[3]:
        words = full2half(words)
    return words


In [46]:

test_words = "aBc 歡迎来到万门大学ｐｙｔｈｏｎ"
cleaning_parameters = [True, True, True, True]
data_cleaning(test_words, cleaning_parameters)


'abc欢迎来到万门大学python'

## 数据分词

In [47]:

class Single_Tokenizer():
    def cut(self, words):
        seg_words = list(words)
        return seg_words


In [48]:
def make_tokenizer(tokenizer_name, userdict_path="", stopwords_path=""):
    """
    选取分词器
    :param tokenizer_name: str 分词器的名称
    :param userdict_path: str 自定义词典的路径
    :param stopwords_path: str 停用词表的路径
    :return: object 分词器返回
    """
    if stopwords_path != "":
        # 将停用词读出来放在stopwords这个列表中
        stopwords = [line.strip() for line in open(stopwords_path, 'r', encoding='utf-8').readlines()]
    else:
        stopwords = []
    if tokenizer_name == "pkuseg":
        if userdict_path != "":
            pku = pkuseg.pkuseg(user_dict=userdict_path)
        else:
            pku = pkuseg.pkuseg()
        return pku, stopwords
    elif tokenizer_name == "single":
        single_tokenizer = Single_Tokenizer()
        return single_tokenizer, stopwords
    else:
        # 默认使用jieba分词
        if userdict_path != "":
            jieba.load_userdict(userdict_path)
        return jieba, stopwords


In [49]:
def word_seg(all_data, tokenizer, stopwords, cleaning_parameters):
    """
    数据分词，分词前进行数据清洗
    :param all_data: list 原始数据
    :param tokenizer: object 分词器
    :param stopwords: list 停用词列表
    :param cleaning_parameters: list 传入的控制开关列表
    :return: list 分完词之后的数据
    """
    segmented_all_data = []
    for sentences, tag in all_data:
        sentences = data_cleaning(sentences, cleaning_parameters)
        seg_list = tokenizer.cut(sentences)
        seg_list = [i for i in seg_list if i not in stopwords]
        segmented_all_data.append([seg_list, tag])
    return segmented_all_data


In [50]:

tokenizer_name = "jieba"
userdict_path = "./userdict/userdict.txt"
stopwords_path = "./stopwords/stopwords.txt"
tokenizer, stopwords = make_tokenizer(tokenizer_name, userdict_path, stopwords_path)


In [51]:
cleaning_parameters = [True, True, True, True]
segmented_all_data = word_seg(all_data[:50000], tokenizer, stopwords, cleaning_parameters)
print(segmented_all_data[:5])


[[['上课时', '学生', '手机', '响个', '不停', '老师', '一怒之下', '把', '手机', '摔', '了', '家长', '拿', '发票', '让', '老师', '赔', '大家', '怎么', '看待', '这种', '事'], 'edu'], [['商赢', '环球', '股份', '有限公司', '关于', '延期', '回复', '上海证券交易所', '对', '公司', '2017', '年', '年度报告', '的', '事后', '审核', '问询', '函', '的', '公告'], 'finance'], [['通过', '中介', '公司', '买', '了', '二手房', '首付', '都', '付', '了', '现在', '卖家', '不想', '卖', '了', '怎么', '处理'], 'house'], [['2018', '年', '去', '俄罗斯', '看', '世界杯', '得花', '多少', '钱'], 'travel'], [['剃须刀', '的', '个性', '革新', '雷明登', '天猫', '定制', '版', '新品', '首发'], 'tech']]


## 生成word2id, tag2id

### 直接生成

In [52]:

def make_map_dict(segmented_all_data):
    """
    制作词到ID的映射，标签到ID的映射
    :param segmented_all_data: list 分完词之后的数据
    :return: dict 词到ID的映射及标签到ID的映射
    """
    all_tag = []
    all_words = []
    all_words.append('<PAD>')
    all_words.append('<UNK>')
    for seg_list, tag in segmented_all_data:
        for word in seg_list:
            if word not in all_words:
                all_words.append(word)
        if tag not in all_tag:
            all_tag.append(tag)
    word2id = {all_words[i]: i for i in range(len(all_words))}
    id2word = {v: k for k, v in word2id.items()}
    tag2id = {all_tag[i]: i for i in range(len(all_tag))}
    id2tag = {v: k for k, v in tag2id.items()}
    return word2id, id2word, tag2id, id2tag


In [53]:

word2id, id2word, tag2id, id2tag = make_map_dict(segmented_all_data)
print(dict(list(word2id.items())[:10])) 

{'<PAD>': 0, '<UNK>': 1, '上课时': 2, '学生': 3, '手机': 4, '响个': 5, '不停': 6, '老师': 7, '一怒之下': 8, '把': 9}


In [54]:
len(word2id)

64854

### 通过tfidf生成

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def make_map_dict(segmented_all_data,tokenizer_name):
    '''
    生成word2id,id2word,tag2id,id2tag
    params:
        segmented_all_data: 分词后的数据
        tokenizer: 分词器
    return:
        word2id,id2word,tag2id,id2tag
    '''
    all_tag = []
    all_words = []
    all_words.append('<PAD>')
    all_words.append('<UNK>')
    for seg_list, tag in segmented_all_data:
        if tag not in all_tag:
            all_tag.append(tag)
    tag2id = {all_tag[i]:i for i in range(len(all_tag))}
    id2tag = {v:k for k,v in tag2id.items()}

    if tokenizer_name != 'single':
        # 如果不是以字为特征，则引入tfidf的初始化帮助清洗数据
        all_texts = [" ".join(i[0]) for i in segmented_all_data]
        tfidf_vec = TfidfVectorizer(max_features=15000, max_df=0.8, min_df=2)
        tfidf_mat = tfidf_vec.fit_transform(all_texts)
        for word in tfidf_vec.vocabulary_.keys():
            if word not in all_words:
                all_words.append(word)
    else:
        # 如果以字为特征，则直接生成word2id,id2word
        for seg_list, tag in segmented_all_data:
            for word in seg_list:
                if word not in all_words:
                    all_words.append(word)
    word2id = {all_words[i]: i for i in range(len(all_words))}
    id2word = {v:k for k,v in word2id.items()}
    return word2id,id2word,tag2id,id2tag
   

In [67]:
tokenizer_name = 'jieba'
word2id,id2word,tag2id,id2tag = make_map_dict(segmented_all_data,tokenizer_name)
print(dict(list(word2id.items())[:10])) 


{'<PAD>': 0, '<UNK>': 1, '王者': 2, '荣耀': 3, '初夏': 4, '活动': 5, '悄然': 6, '开启': 7, '夏日': 8, '皮肤': 9}


In [58]:
print(len(word2id))

15002


## 数据切分

In [59]:
random.seed(1)
random.shuffle(segmented_all_data)
data_len = len(segmented_all_data)
train_ratio = int(data_len * 0.8)
train_data = segmented_all_data[:train_ratio]
test_data = segmented_all_data[train_ratio:]

len(test_data)

10000

In [60]:
len(train_data)

40000

## 数据解析

In [61]:
# 词>词id>通过id去找vec

In [72]:
def parse_data(data, max_len,word2id,tag2id):
    parsed_data = []
    for sentences, tag in data:
        sent_id = [word2id[word] if word in word2id else word2id["<UNK>"] for word in sentences]
        tag_id = tag2id[tag]
        if len(sent_id) > max_len:
            sent_id = sent_id[:max_len]
        else:
            sent_id = sent_id + [word2id["<PAD>"]] * (max_len - len(sent_id))
        parsed_data.append([sent_id, tag_id])
    return parsed_data

In [74]:
max_len = 50
all_seq = parse_data(segmented_all_data, max_len,word2id,tag2id)
print(all_seq[:5])

[[[2, 3, 4, 5, 6, 7, 8, 9, 1, 10, 11, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 0], [[12, 13, 1, 1, 1, 14, 15, 16, 17, 18, 1, 19, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 1], [[20, 21, 22, 23, 24, 25, 1, 1, 1, 26, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 2], [[27, 1, 28, 29, 30, 1, 31, 1, 32, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 3], [[2, 3, 1, 33, 34, 35, 36, 1, 1, 1, 1, 37, 38, 39, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 0]]


## 数据生成器

In [80]:
def get_batch(data,batch_size,shuffle=False):
    """
    数据生成器，将数据切块，供神经网络训练
    params:
        data: 原始数据
        batch_size: 每个批次的样本数
        shuffle: 是否打乱数据顺序
    return:
        tuple 每一块数据
    """
    if shuffle:
        random.shuffle(data)
    for i in range(0,len(data),batch_size):
        data_batch = data[i:i+batch_size]
        inputseqs, input_labels = [], []
        for sent_id, tag_id in data_batch:
            inputseqs.append(sent_id)
            input_labels.append(tag_id)
        yield np.array(inputseqs), np.array(input_labels)


In [81]:
batch_size = 128
shuffle = True
for inputseqs, input_labels in get_batch(all_seq, batch_size, shuffle):
    print(inputseqs.shape, input_labels.shape)
    break

(128, 50) (128,)
